In [25]:
import pennylane as qml
import numpy as np
from pennylane import numpy as pnp
from matplotlib import pyplot as plt
from pennylane.operation import Operation, AnyWires
import os
import pandas as pd
import scipy.special as sp
import math
from scipy.stats import uniform_direction
from scipy.linalg import logm, svd

num_qubits = 5

# Initialize the device
dev = qml.device("lightning.qubit", wires=num_qubits)
dev2 = qml.device("lightning.qubit", wires=1)

In [ ]:
# Construct the Hamiltonian terms
Hamiltonian_terms = []

# Interaction terms: XiX(i+1) + YiY(i+1) + ZiZ(i+1)
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * (qml.PauliX(i) @ qml.PauliX((i+1)%num_qubits)) +
                                    (qml.PauliY(i) @ qml.PauliY((i+1)%num_qubits)) 
                                    + (qml.PauliZ(i) @ qml.PauliZ((i+1)%num_qubits)))

# Magnetic field terms: hZi
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * qml.PauliZ(i))

# Define the Hamiltonian
Hamiltonian = qml.Hamiltonian(coeffs=[1] * len(Hamiltonian_terms), observables=Hamiltonian_terms)

In [29]:
# define identity matrix and pauli matrices

Id = np.eye(2)

x = np.matrix([[0, 1.0],
               [1.0, 0]])

y = np.matrix([[0, -1.0j],
               [1.0j, 0]])

z = np.matrix([[1.0, 0],
               [0, -1.0]])

In [ ]:

def entangling_layer_ladderZ(num_qubits):
    m = 0
    n = 1
    while m+1 < num_qubits:
        qml.CZ(wires=[m,m+1])
        m+=2
    
    while n+1 < num_qubits:
        qml.CZ(wires=[n,n+1])
        n+=2


@qml.qnode(dev)
def circuit(n_vectors, num_layers):
    """Parameterized quantum circuit with unit quaternions"""
    
    for j in range(num_layers):
        for k in range(num_qubits):
            q0, q1, q2, q3 = n_vectors[k + num_qubits * j]
            unitary = -1j * (x * q1 + y* q2 + z*q3) + Id*q0
            qml.QubitUnitary(unitary, wires = k)
    
        entangling_layer_ladderZ(num_qubits)

    return qml.expval(Hamiltonian)

@qml.qnode(dev)
def circuit_state(n_vectors, num_layers, d, gate_type):
    """Parameterized quantum circuit with unit quaternions"""
    
    ind = 0

    for j in range(num_layers):
    
        for k in range(num_qubits):

            if ind == d:
                if gate_type == "X":
                    qml.QubitUnitary(x, wires = k)

                elif gate_type == "Y":
                    qml.QubitUnitary(y, wires = k)

                elif gate_type == "Z":
                    qml.QubitUnitary(z, wires = k)

                elif gate_type == "XY":
                    unitary = (x + y) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
   
                elif gate_type == "XZ":
                    unitary = (x + z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

                elif gate_type == "YZ":
                    unitary = (y + z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
                
                elif gate_type == "I":
                    qml.Identity(wires = k)

                elif gate_type == "I_X":
                    unitary = (-1j*x + Id) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

                elif gate_type == "I_Y":
                    unitary = (-1j*y + Id) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
                
                elif gate_type == "I_Z":
                    unitary = (Id -1j*z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

            else:
                q0, q1, q2, q3 = n_vectors[k + num_qubits * j]
                unitary = -1j * (x * q1 + y* q2 + z*q3) + Id*q0
                qml.QubitUnitary(unitary, wires = k)

            ind += 1
    
        entangling_layer_ladderZ(num_qubits)

    
    return qml.expval(Hamiltonian)


In [ ]:

def compute_fqs_matrix(n_vectors, num_layers, d):
    """Compute the FQS matrix for a specific gate d"""

    rx = circuit_state(n_vectors, num_layers, d, gate_type="X")
    ry = circuit_state(n_vectors, num_layers, d, gate_type="Y")
    rz = circuit_state(n_vectors, num_layers, d, gate_type="Z")
    
    rxy = circuit_state(n_vectors, num_layers, d, gate_type="XY")
    rxz = circuit_state(n_vectors, num_layers, d, gate_type="XZ")
    ryz = circuit_state(n_vectors, num_layers, d, gate_type="YZ")
    
    Id = circuit_state(n_vectors, num_layers, d, gate_type="I")
    Id_x = circuit_state(n_vectors, num_layers, d, gate_type="I_X")
    Id_y = circuit_state(n_vectors, num_layers, d, gate_type="I_Y")
    Id_z = circuit_state(n_vectors, num_layers, d, gate_type="I_Z")
    
    #print(Id_z)


    matrix = [[Id             , Id_x-rx/2-Id/2, Id_y-ry/2-Id/2, Id_z-rz/2-Id/2],
                [Id_x-rx/2-Id/2,  rx            , (2*rxy-rx-ry)/2, (2*rxz-rx-rz)/2],
                [Id_y-ry/2-Id/2, (2*rxy-rx-ry)/2,  ry            , (2*ryz-ry-rz)/2],
                [Id_z-rz/2-Id/2, (2*rxz-rx-rz)/2, (2*ryz-ry-rz)/2,  rz            ]]

    return matrix



In [32]:
Id = np.matrix([[1,0],
               [0,1]])

X = np.matrix([[0,1],
               [1,0]])

Y = np.matrix([[0,-1j],
               [1j,0]])

Z = np.matrix([[1,0],
               [0,-1]])

In [ ]:

def quaternion_to_unitary(q):
    U = q[0] * Id - 1j * q[1] * X - 1j * q[2] * Y - 1j * q[3] * Z
    return U


def is_unitary(U):
    return np.allclose(U.conj().T @ U, np.eye(U.shape[0]))


def bloch_dist(U, V):
    inner_product = np.trace(U.conj().T @ V)
    dist = np.sqrt(2.0 - np.abs(inner_product))
    
    return dist


In [34]:


def free_quaternion_selection(n_vectors, num_layers, iters, freeze_threshold, freeze_iters_k):
    """Implement the Fraxis algorithm"""

    num_gates = len(n_vectors)

    all_vals = []

    all_dists = []

    freeze_counters = np.zeros(len(n_vectors))    
    
    gate_opts_tresh = num_qubits * num_layers * iters 
    gate_opts = 0
    
    for i in range(iters):

        dists = []

        for d in range(num_gates):

            prev_q = np.array(n_vectors[d].copy())

            current_val = circuit(n_vectors, num_layers)

            # Use this for ideal simulator
            if len(all_vals) > 0:
                if current_val > all_vals[-1] or current_val < min(Hamiltonian.eigvals()):
                    all_vals.append(all_vals[-1])
                else:
                    all_vals.append(current_val)
            else:
                all_vals.append(current_val)
            
            fqs_matrix = compute_fqs_matrix(n_vectors, num_layers, d)        

            eigVal, eigVec = np.linalg.eig(fqs_matrix)
            eigVec = np.transpose(eigVec)

            sid = np.argmin(eigVal)
            expected_val = np.amin(eigVal)
            
            if expected_val < current_val:
                n_vectors[d]  = eigVec[sid]

            current_q = np.array(n_vectors[d].copy())

            prev_unitary =  quaternion_to_unitary(prev_q)
            current_unitary = quaternion_to_unitary(current_q)

            bloch_dist_normalized = bloch_dist(prev_unitary, current_unitary) / np.sqrt(2.0)

            dists.append(bloch_dist_normalized)
        
        all_dists.append(np.array(dists))

    return n_vectors, all_vals, freeze_iters_k, all_dists



In [ ]:
# Initialize parameters and run optimization
layers = [5*num_qubits]
iters = 10

trials = 20

d_vals = [0.025, 0.01, 0.005]

uniform_sphere_dist = uniform_direction(4)

for d_val in d_vals:
    # freeze threshold as the angle 
    freeze_threshold = d_val
    
    for num_layers in layers:
        for trial in range(trials):
            print("trials", trial+1)
            num_gates = num_qubits * num_layers
            freeze_iters = np.ones(num_gates)

            n_vectors = uniform_sphere_dist.rvs(num_gates)
            
            optimal_n_vectors, opt_vals, freeze_iters_k, all_dists = free_quaternion_selection(n_vectors, num_layers, iters, freeze_threshold, freeze_iters)
            
            file2 = f"BlochDist_1DHeisenberg_{num_qubits}Q_FQS_GateFreeze_d{d_val}_FreezeIterInc_{iters}cycles_{num_layers}layers_{trials}trials_A.xlsx"
            
            if not os.path.exists(file2):
                df2 = pd.DataFrame()
                df2.to_excel(file2)

            df2 = pd.read_excel(file2)

            if len(df2.columns) < trials:
                
                df2[f"col{len(df2.columns)}"] = pd.Series(opt_vals)
                df2.to_excel(file2,index = False)
            else:
                break
                



trials 1
trials 2


C:\Users\joonp\AppData\Local\Temp\ipykernel_36760\2268920664.py:43: ComplexWarning: Casting complex values to real discards the imaginary part
  n_vectors[d]  = eigVec[sid]


In [ ]:
print(qml.draw(circuit)(n_vectors, num_layers))


0: ──U(M0)─╭●──U(M5)────────╭●──U(M10)─────────╭●──U(M15)─────────╭●──U(M20)─────────╭●──U(M25)
1: ──U(M1)─╰Z─╭●──────U(M6)─╰Z─╭●───────U(M11)─╰Z─╭●───────U(M16)─╰Z─╭●───────U(M21)─╰Z─╭●─────
2: ──U(M2)─╭●─╰Z──────U(M7)─╭●─╰Z───────U(M12)─╭●─╰Z───────U(M17)─╭●─╰Z───────U(M22)─╭●─╰Z─────
3: ──U(M3)─╰Z─╭●──────U(M8)─╰Z─╭●───────U(M13)─╰Z─╭●───────U(M18)─╰Z─╭●───────U(M23)─╰Z─╭●─────
4: ──U(M4)────╰Z──────U(M9)────╰Z───────U(M14)────╰Z───────U(M19)────╰Z───────U(M24)────╰Z─────

──────────╭●──U(M30)─────────╭●──U(M35)─────────╭●──U(M40)─────────╭●──U(M45)─────────╭●──U(M50)
───U(M26)─╰Z─╭●───────U(M31)─╰Z─╭●───────U(M36)─╰Z─╭●───────U(M41)─╰Z─╭●───────U(M46)─╰Z─╭●─────
───U(M27)─╭●─╰Z───────U(M32)─╭●─╰Z───────U(M37)─╭●─╰Z───────U(M42)─╭●─╰Z───────U(M47)─╭●─╰Z─────
───U(M28)─╰Z─╭●───────U(M33)─╰Z─╭●───────U(M38)─╰Z─╭●───────U(M43)─╰Z─╭●───────U(M48)─╰Z─╭●─────
───U(M29)────╰Z───────U(M34)────╰Z───────U(M39)────╰Z───────U(M44)────╰Z───────U(M49)────╰Z─────

──────────╭●──U(M55)─────────╭●──